# LeetCode #81 — Search in Rotated Sorted Array II: binary search khi có phần tử trùng

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search

Skeleton trên LeetCode:

```python
class Solution:
    def search(self, nums: List[int], target: int) -> bool:
```

Bài này là phiên bản có **phần tử trùng** của #33. Mục tiêu của notebook:

1. Ôn lại ý tưởng #33: một trong hai nửa quanh `mid` luôn **đã sắp xếp**.
2. Thấy vì sao phần tử trùng phá vỡ ý tưởng đó (`nums[lo] == nums[mid] == nums[hi]`).
3. Xử lý bằng cách **thu hẹp `lo += 1`, `hi -= 1`** và hiểu cái giá: trường hợp xấu `O(n)`.
4. Trả lời follow-up của đề: trùng lặp ảnh hưởng thế nào tới độ phức tạp.

# 1. Bài toán là gì?

Mảng `nums` sắp xếp **không giảm** (có thể có phần tử trùng), rồi bị **xoay** tại một chỉ số `k` chưa biết: `[nums[k], …, nums[n-1], nums[0], …, nums[k-1]]`.

Ví dụ `[0,1,2,4,4,4,5,6,6,7]` xoay tại `k = 5` thành `[4,5,6,6,7,0,1,2,4,4]`.

Cho `target`, trả `True` nếu có trong `nums`. Cần giảm số bước nhiều nhất có thể.

**Constraints (theo đề):**

- `1 <= nums.length <= 5000`
- `-10^4 <= nums[i], target <= 10^4`
- `nums` được đảm bảo là mảng sắp xếp bị xoay.

**Follow up (theo đề):** bài này giống #33 nhưng `nums` có thể có trùng. Điều đó có ảnh hưởng tới độ phức tạp không? Vì sao?

---

## Ví dụ

```text
nums = [2, 5, 6, 0, 0, 1, 2], target = 0  -> True
nums = [2, 5, 6, 0, 0, 1, 2], target = 3  -> False
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Áp dụng vào bài này

**Tìm nhị phân** trên mảng xoay như #33, nhưng có **trùng**: khi `nums[lo] == nums[mid] == nums[hi]` không biết nửa nào đã sắp xếp → thu hẹp `lo += 1, hi -= 1`; trường hợp xấu thành `O(n)`.

# 3. Ôn lại #33 (không trùng)

Lấy `mid`. Mảng xoay gồm hai đoạn tăng. `mid` nằm trong một đoạn, nên:

- Nếu `nums[lo] <= nums[mid]` → nửa **trái** `[lo, mid]` đã sắp xếp.
- Ngược lại → nửa **phải** `[mid, hi]` đã sắp xếp.

Trên nửa đã sắp xếp, kiểm tra `target` có nằm trong khoảng giá trị của nó không → biết bỏ nửa nào.

In [4]:
from typing import List
import random

def search_33(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] <= nums[mid]:                     # trái đã sắp xếp
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                         # phải đã sắp xếp
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False

print(search_33([4, 5, 6, 7, 0, 1, 2], 0), search_33([4, 5, 6, 7, 0, 1, 2], 3))

True False


# 4. Phần tử trùng phá vỡ điều gì?

Xét `nums = [1, 0, 1, 1, 1]`, `target = 0`. Lần đầu `lo = 0, mid = 2, hi = 4`: `nums[lo] = nums[mid] = nums[hi] = 1`.

`nums[lo] <= nums[mid]` đúng, nên bản #33 kết luận "nửa trái `[1, 0, 1]` đã sắp xếp" — **sai**! Rồi `target = 0` không nằm trong `[1, 1)` → bỏ nửa trái → mất số `0`.

In [5]:
print("Bản #33 trên [1, 0, 1, 1, 1], target 0:", search_33([1, 0, 1, 1, 1], 0), "(đúng phải là True)")
print("Bản #33 trên [1, 1, 1, 0, 1], target 0:", search_33([1, 1, 1, 0, 1], 0))

Bản #33 trên [1, 0, 1, 1, 1], target 0: False (đúng phải là True)
Bản #33 trên [1, 1, 1, 0, 1], target 0: True


Hai mảng `[1, 0, 1, 1, 1]` và `[1, 1, 1, 0, 1]` có **cùng** `nums[lo], nums[mid], nums[hi]`, nhưng số `0` nằm ở hai nửa khác nhau. Không có cách nào biết bỏ nửa nào chỉ từ ba giá trị đó.

# 5. Ý tưởng chính: khi không phân biệt được, thu hẹp hai đầu

Nếu `nums[lo] == nums[mid] == nums[hi]` (và khác `target`, vì đã kiểm tra `nums[mid]`), thì `nums[lo]` và `nums[hi]` **không phải** `target` → bỏ chúng an toàn:

```text
lo += 1
hi -= 1
```

Các trường hợp còn lại giống #33. (Thực ra chỉ cần điều kiện `nums[lo] == nums[mid]`: khi đó không chắc nửa trái đã sắp xếp, và `nums[lo] != target` nên `lo += 1` là an toàn.)

In [6]:
def search_dup(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False

print(search_dup([1, 0, 1, 1, 1], 0), search_dup([1, 1, 1, 0, 1], 0), search_dup([2, 5, 6, 0, 0, 1, 2], 3))

True True False


**Vì sao khi chỉ `nums[lo] == nums[mid]` (còn `nums[hi]` khác) thì nhánh `nums[lo] <= nums[mid]` vẫn đúng?** Nếu `nums[mid] == nums[lo]` mà `nums[hi] != nums[lo]`, thì điểm xoay không thể nằm trong `(lo, mid]` (nếu không, mọi giá trị từ `lo` tới điểm xoay, qua cả đoạn sau, sẽ phải bằng nhau và kéo theo `nums[hi] == nums[lo]`). Nên `[lo, mid]` thật sự đã sắp xếp (toàn giá trị bằng nhau).

# 6. Mô phỏng

In [7]:
def trace_search(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    print(f"nums = {nums}, target = {target}")
    while lo <= hi:
        mid = (lo + hi) // 2
        s = f"  lo={lo} mid={mid} hi={hi} | {nums[lo]}, {nums[mid]}, {nums[hi]}"
        if nums[mid] == target:
            print(s, "-> nums[mid] == target -> True")
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            print(s, "-> ba giá trị bằng nhau -> lo++, hi--")
            lo += 1; hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                print(s, "-> trái sắp xếp, target trong trái -> hi = mid-1")
                hi = mid - 1
            else:
                print(s, "-> trái sắp xếp, target không trong trái -> lo = mid+1")
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                print(s, "-> phải sắp xếp, target trong phải -> lo = mid+1")
                lo = mid + 1
            else:
                print(s, "-> phải sắp xếp, target không trong phải -> hi = mid-1")
                hi = mid - 1
    print("  lo > hi -> False")
    return False

_ = trace_search([2, 5, 6, 0, 0, 1, 2], 0)
print()
_ = trace_search([1, 0, 1, 1, 1], 0)

nums = [2, 5, 6, 0, 0, 1, 2], target = 0
  lo=0 mid=3 hi=6 | 2, 0, 2 -> nums[mid] == target -> True

nums = [1, 0, 1, 1, 1], target = 0
  lo=0 mid=2 hi=4 | 1, 1, 1 -> ba giá trị bằng nhau -> lo++, hi--
  lo=1 mid=2 hi=3 | 0, 1, 1 -> trái sắp xếp, target trong trái -> hi = mid-1
  lo=1 mid=1 hi=1 | 0, 0, 0 -> nums[mid] == target -> True


# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def search(self, nums: List[int], target: int) -> bool:
        lo, hi = 0, len(nums) - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target:
                return True
            if nums[lo] == nums[mid] == nums[hi]:
                lo += 1
                hi -= 1
            elif nums[lo] <= nums[mid]:
                if nums[lo] <= target < nums[mid]:
                    hi = mid - 1
                else:
                    lo = mid + 1
            else:
                if nums[mid] < target <= nums[hi]:
                    lo = mid + 1
                else:
                    hi = mid - 1
        return False

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


def rotated(arr, k):
    return arr[k:] + arr[:k]


sol = Solution()
check("Ví dụ 1", sol.search([2, 5, 6, 0, 0, 1, 2], 0), True)
check("Ví dụ 2", sol.search([2, 5, 6, 0, 0, 1, 2], 3), False)
check("Bẫy trùng (trái)", sol.search([1, 0, 1, 1, 1], 0), True)
check("Bẫy trùng (phải)", sol.search([1, 1, 1, 0, 1], 0), True)
check("Một phần tử", sol.search([1], 0), False)
check("Toàn giống nhau", sol.search([3, 3, 3, 3], 3), True)
check("Không xoay", sol.search([1, 2, 2, 3], 3), True)

random.seed(81)
for _ in range(20000):
    base = sorted(random.randint(0, 4) for _ in range(random.randint(1, 10)))
    nums = rotated(base, random.randint(0, len(base) - 1))
    t = random.randint(-1, 5)
    assert sol.search(nums, t) == (t in nums), (nums, t)
print("20000 mảng xoay ngẫu nhiên (nhiều trùng): khớp toán tử `in`")

Ví dụ 1                    PASS  True
Ví dụ 2                    PASS  False
Bẫy trùng (trái)           PASS  True
Bẫy trùng (phải)           PASS  True
Một phần tử                PASS  False
Toàn giống nhau            PASS  True
Không xoay                 PASS  True


20000 mảng xoay ngẫu nhiên (nhiều trùng): khớp toán tử `in`


# 8. Vì sao đúng? Độ phức tạp? (trả lời follow-up)

**Đúng:** mỗi bước chỉ bỏ những vị trí chắc chắn **không** chứa `target`:

- `lo += 1; hi -= 1` khi `nums[lo] == nums[hi] == nums[mid] != target`.
- Các nhánh còn lại: nửa được chọn là nửa đã sắp xếp thật (lý luận ở mục 5), nên kiểm tra khoảng giá trị là chính xác.

**Độ phức tạp:**

```text
Trung bình (ít trùng): O(log n)
Trường hợp xấu:        O(n)     ví dụ [1, 1, ..., 1, 0, 1, ..., 1] (số 0 gần giữa)
Space:                 O(1)
```

**Follow-up:** Có, trùng lặp làm trường hợp xấu thành `O(n)`. Và **không thể** tốt hơn: với mảng toàn số `1` chỉ có **một** số `0` ở vị trí bất kỳ, mọi thuật toán (so sánh) đều phải xem gần hết các phần tử mới biết số `0` ở đâu — vì đọc bất kỳ vị trí nào khác cũng chỉ thấy `1`.

In [10]:
def count_steps(nums, target):
    steps = 0
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return steps
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1; hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]: hi = mid - 1
            else: lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]: lo = mid + 1
            else: hi = mid - 1
    return steps

n = 5000
distinct = rotated(list(range(n)), 1234)
ones = [1] * n; ones[n // 2 + 1] = 0          # một số 0 nằm ngay sau giữa mảng
print("Không trùng, n = 5000:", count_steps(distinct, 77), "bước")
print("Toàn số 1 trừ một số 0, n = 5000:", count_steps(ones, 0), "bước")

Không trùng, n = 5000: 11 bước
Toàn số 1 trừ một số 0, n = 5000: 2501 bước


# 9. Những lỗi dễ mắc

## Lỗi 1: Dùng nguyên code #33

Như mục 4 — kết luận sai nửa nào đã sắp xếp khi `nums[lo] == nums[mid]`.

## Lỗi 2: Chỉ `lo += 1` nhưng vẫn rơi vào nhánh "trái sắp xếp" ở cùng vòng

Phải dùng `if / elif / else` để sau khi thu hẹp thì **sang vòng mới**, không xử lý tiếp với `mid` cũ.

In [11]:
def wrong_no_elif(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        if nums[lo] <= nums[mid]:                        # SAI: thiếu elif / continue
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False

print("Sai :", wrong_no_elif([1, 1, 1, 0, 1, 1, 1, 1, 1], 0), "| Đúng:", sol.search([1, 1, 1, 0, 1, 1, 1, 1, 1], 0))

Sai : False | Đúng: True


## Lỗi 3: Dấu so sánh ở biên khoảng

Nửa trái sắp xếp `[lo, mid]` và `nums[mid] != target`: `target` trong trái ⇔ `nums[lo] <= target < nums[mid]` (bao `lo`, không bao `mid`). Viết `nums[lo] < target` sẽ bỏ sót `target == nums[lo]`.

## Lỗi 4: Nghĩ rằng vẫn `O(log n)`

Trong phỏng vấn, follow-up là câu hỏi chính: phải nói được trường hợp xấu `O(n)` và vì sao không tránh được.

# 10. Mẹo nhận diện pattern

**Binary search trên mảng xoay:** luôn tìm "nửa đã sắp xếp" rồi kiểm tra `target` có trong khoảng giá trị của nửa đó.

| Bài | Có trùng? | Kỹ thuật thêm |
|---|---|---|
| #33 Search in Rotated Sorted Array | không | — |
| #81 (bài này) | có | `lo++, hi--` khi ba đầu bằng nhau |
| #153 Find Minimum in Rotated Sorted Array | không | so `nums[mid]` với `nums[hi]` |
| #154 Find Minimum … II (Hard) | có | `hi -= 1` khi `nums[mid] == nums[hi]` |

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Mô phỏng thuật toán cho `nums = [3, 1, 2, 3, 3, 3, 3]`, `target = 2`.

## Bài 2 — Điền chỗ trống

In [12]:
def exercise_2_search(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # TODO: kiểm tra mid; trường hợp ba đầu bằng nhau; nửa trái sắp xếp; nửa phải sắp xếp
        pass
    return False


# assert exercise_2_search([2, 5, 6, 0, 0, 1, 2], 0) is True
# assert exercise_2_search([2, 5, 6, 0, 0, 1, 2], 3) is False
# assert exercise_2_search([1, 0, 1, 1, 1], 0) is True

## Bài 3 — Biến thể: Find Minimum in Rotated Sorted Array II (LeetCode #154)

Mảng xoay có thể có trùng. Tìm giá trị nhỏ nhất.

```python
find_min([1, 3, 5])        -> 1
find_min([2, 2, 2, 0, 1])  -> 0
```

Gợi ý nhanh: so `nums[mid]` với `nums[hi]`: lớn hơn → min ở phải (`lo = mid + 1`); nhỏ hơn → min ở `[lo, mid]` (`hi = mid`); bằng → `hi -= 1`.

In [13]:
def exercise_3_find_min(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_find_min([1, 3, 5]) == 1
# assert exercise_3_find_min([2, 2, 2, 0, 1]) == 0
# assert exercise_3_find_min([3, 3, 1, 3]) == 1

## Bài 4 — Biến thể: chỉ số đầu tiên của `target` (không trùng)

Với mảng xoay **không trùng** (#33), trả chỉ số của `target` hoặc `-1`.

In [14]:
def exercise_4_search_index(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_4_search_index([4, 5, 6, 7, 0, 1, 2], 0) == 4
# assert exercise_4_search_index([4, 5, 6, 7, 0, 1, 2], 3) == -1
# assert exercise_4_search_index([1], 1) == 0

## Bài 5 — Mini challenge: tìm điểm xoay `k` (không trùng) rồi binary search thường

Cách khác cho #33: tìm chỉ số phần tử nhỏ nhất `k` (= số bước đã xoay) bằng binary search, rồi tìm `target` trên mảng "ảo" `nums[(i + k) % n]` đã sắp xếp. Trả chỉ số trong `nums` hoặc `-1`.

In [15]:
def exercise_5_search_via_pivot(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_5_search_via_pivot([4, 5, 6, 7, 0, 1, 2], 0) == 4
# assert exercise_5_search_via_pivot([4, 5, 6, 7, 0, 1, 2], 7) == 3
# assert exercise_5_search_via_pivot([1, 3], 3) == 1
# assert exercise_5_search_via_pivot([5, 1, 3], 4) == -1

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_search` sau khi tự làm.

**Bài 3**
- Mức 1: `while lo < hi` (nửa mở), trả `nums[lo]`.

**Bài 4**
- Mức 1: như `search_33`, nhưng trả `mid` thay vì `True`.

**Bài 5**
- Mức 1: tìm `k`: `lo, hi = 0, n-1`; `while lo < hi`: nếu `nums[mid] > nums[hi]` thì `lo = mid + 1` else `hi = mid`.
- Mức 2: binary search `i` trong `[0, n-1]` với giá trị `nums[(i + k) % n]`; tìm thấy thì trả `(i + k) % n`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [16]:
_ = trace_search([3, 1, 2, 3, 3, 3, 3], 2)

nums = [3, 1, 2, 3, 3, 3, 3], target = 2
  lo=0 mid=3 hi=6 | 3, 3, 3 -> ba giá trị bằng nhau -> lo++, hi--
  lo=1 mid=3 hi=5 | 1, 3, 3 -> trái sắp xếp, target trong trái -> hi = mid-1
  lo=1 mid=1 hi=2 | 1, 1, 2 -> trái sắp xếp, target không trong trái -> lo = mid+1
  lo=2 mid=2 hi=2 | 2, 2, 2 -> nums[mid] == target -> True


## Đáp án Bài 2

In [17]:
def answer_2_search(nums: List[int], target: int) -> bool:
    return search_dup(nums, target)

assert answer_2_search([2, 5, 6, 0, 0, 1, 2], 0) is True
assert answer_2_search([2, 5, 6, 0, 0, 1, 2], 3) is False
assert answer_2_search([1, 0, 1, 1, 1], 0) is True
print("Bài 2 PASS (giống search_dup ở mục 5)")

Bài 2 PASS (giống search_dup ở mục 5)


## Đáp án Bài 3

In [18]:
def answer_3_find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid
        else:
            hi -= 1
    return nums[lo]

assert answer_3_find_min([1, 3, 5]) == 1
assert answer_3_find_min([2, 2, 2, 0, 1]) == 0
assert answer_3_find_min([3, 3, 1, 3]) == 1
for _ in range(5000):
    base = sorted(random.randint(0, 4) for _ in range(random.randint(1, 10)))
    nums = rotated(base, random.randint(0, len(base) - 1))
    assert answer_3_find_min(nums) == min(nums)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [19]:
def answer_4_search_index(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

assert answer_4_search_index([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert answer_4_search_index([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert answer_4_search_index([1], 1) == 0
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [20]:
def answer_5_search_via_pivot(nums: List[int], target: int) -> int:
    n = len(nums)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    k = lo
    lo, hi = 0, n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        real = (mid + k) % n
        if nums[real] == target:
            return real
        if nums[real] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

assert answer_5_search_via_pivot([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert answer_5_search_via_pivot([4, 5, 6, 7, 0, 1, 2], 7) == 3
assert answer_5_search_via_pivot([1, 3], 3) == 1
assert answer_5_search_via_pivot([5, 1, 3], 4) == -1
for _ in range(5000):
    base = sorted(random.sample(range(-20, 20), random.randint(1, 10)))
    nums = rotated(base, random.randint(0, len(base) - 1))
    t = random.randint(-21, 21)
    expected = nums.index(t) if t in nums else -1
    assert answer_5_search_via_pivot(nums, t) == expected == answer_4_search_index(nums, t)
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Như #33: tìm nửa đã sắp xếp và kiểm tra khoảng giá trị. Khi `nums[lo] == nums[mid] == nums[hi]` thì không biết nửa nào sắp xếp → `lo += 1, hi -= 1`. Trung bình `O(log n)`, xấu nhất `O(n)` — và không thể tránh được.

---

## Checklist trước khi nộp

- [ ] Kiểm tra `nums[mid] == target` trước?
- [ ] Có nhánh `lo++, hi--` cho trường hợp ba đầu bằng nhau, dùng `elif`?
- [ ] Biên khoảng: `nums[lo] <= target < nums[mid]` và `nums[mid] < target <= nums[hi]`?
- [ ] Trả lời được follow-up về `O(n)`?